# LeetCode 1066 — Campus Bikes II bằng Python

Link đề: https://leetcode.com/problems/campus-bikes-ii/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Bitmask, Backtracking, Hungarian Algorithm, Min-Cost Flow

> **Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết và theo bản mô tả trong kho tham khảo; nếu bạn có tài khoản Premium hãy **đối chiếu lại** đề gốc (đặc biệt về ràng buộc `1 <= n <= m <= 10`, khoảng cách Manhattan và việc mỗi xe chỉ giao cho một công nhân).

Skeleton LeetCode:

```python
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**: tổng khoảng cách Manhattan **nhỏ nhất** khi gán cho mỗi công nhân một xe **khác nhau**. Đây là bài toán **gán (assignment problem)** với `n ≤ m ≤ 10`: DP bitmask trên tập xe đã dùng là cách chuẩn; thuật toán Hungarian cho đáp án đa thức.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` công nhân, `m ≥ n` xe, mỗi công nhân một xe riêng, tối thiểu hoá tổng khoảng cách Manhattan.
2. Nắm công cụ: ma trận chi phí, hoán vị/chỉnh hợp, DP bitmask, nhánh cận (branch and bound), thuật toán Hungarian.
3. Đi từ **vét cạn chỉnh hợp `m!/(m-n)!`** → **quay lui cắt nhánh** → **DP bitmask** → **đệ quy có nhớ** → **Hungarian `O(n²m)`**.
4. Hiểu mấu chốt: xử lý công nhân **theo thứ tự**, trạng thái chỉ cần **tập xe đã dùng** `mask` (và số công nhân đã gán = `popcount(mask)`), nên có `2^m` trạng thái thay vì `m!` hoán vị.
5. Tránh lỗi kinh điển: tham lam gán xe gần nhất (sai), dùng lại một xe cho nhiều người, đòi dùng hết mọi xe khi `n < m`, hằng `INF` quá nhỏ, nhầm khoảng cách (Manhattan so với Chebyshev/Euclid).

## 2. Đề bài & ví dụ

Trên lưới 2D có `n` công nhân và `m` xe đạp (`n ≤ m`), mỗi công nhân và mỗi xe là một toạ độ. Ta gán cho **mỗi công nhân một xe riêng** sao cho **tổng khoảng cách Manhattan** giữa mỗi công nhân và xe được gán là **nhỏ nhất**. Trả về tổng nhỏ nhất đó.

Khoảng cách Manhattan: `|p1.x − p2.x| + |p1.y − p2.y|`.

**Ví dụ 1:** `workers = [[0,0],[2,1]]`, `bikes = [[1,2],[3,3]]` → `6` (công nhân 0 – xe 0 cách `3`, công nhân 1 – xe 1 cách `3`).

**Ví dụ 2:** `workers = [[0,0],[1,1],[2,0]]`, `bikes = [[1,0],[2,2],[2,1]]` → `4`.

**Ví dụ 3:** `workers = [[0,0],[1,0],[2,0],[3,0],[4,0]]`, `bikes = [[0,999],[1,999],[2,999],[3,999],[4,999]]` → `4995`.

**Ràng buộc:** `1 <= n <= m <= 10`; toạ độ trong `[0, 1000)`; mọi vị trí công nhân và xe **phân biệt**.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [
    ([[0, 0], [2, 1]], [[1, 2], [3, 3]], 6),
    ([[0, 0], [1, 1], [2, 0]], [[1, 0], [2, 2], [2, 1]], 4),
    ([[0, 0], [1, 0], [2, 0], [3, 0], [4, 0]], [[0, 999], [1, 999], [2, 999], [3, 999], [4, 999]], 4995),
]
for w, b, want in examples:
    print(f"workers = {w}, bikes = {b} -> {want}")

workers = [[0, 0], [2, 1]], bikes = [[1, 2], [3, 3]] -> 6
workers = [[0, 0], [1, 1], [2, 0]], bikes = [[1, 0], [2, 2], [2, 1]] -> 4
workers = [[0, 0], [1, 0], [2, 0], [3, 0], [4, 0]], bikes = [[0, 999], [1, 999], [2, 999], [3, 999], [4, 999]] -> 4995


## 3. Kiến thức nền

### 3.1 Ma trận chi phí

Đặt `cost[i][j] = |w_i.x − b_j.x| + |w_i.y − b_j.y|`. Một cách gán là một **đơn ánh** `σ: {0..n-1} → {0..m-1}` (hai công nhân khác nhau nhận xe khác nhau); chi phí `Σ cost[i][σ(i)]`. Có `m·(m-1)·…·(m-n+1)` đơn ánh (chỉnh hợp).

In [2]:
def manhattan(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])


def cost_matrix(workers, bikes):
    return [[manhattan(w, b) for b in bikes] for w in workers]


for w, b, _ in examples[:2]:
    print(cost_matrix(w, b))

[[3, 6], [2, 3]]
[[1, 4, 3], [1, 2, 1], [1, 2, 1]]


### 3.2 Bitmask

Một tập con của `m` xe lưu bằng số nguyên `mask` (bit `j` bật nếu xe `j` đã được gán). `mask & (1 << j)` kiểm tra xe `j` đã dùng; `mask | (1 << j)` thêm xe `j`; `bin(mask).count("1")` đếm số xe đã dùng.

### 3.3 Vì sao trạng thái chỉ cần `mask`

Nếu gán công nhân **theo thứ tự `0, 1, 2, …`** thì sau khi đã gán `k` công nhân, tập xe đã dùng có đúng `k` bit. Công nhân kế tiếp là `k = popcount(mask)`. Chi phí tương lai chỉ phụ thuộc vào **xe còn trống** (và công nhân còn lại), không phụ thuộc xe nào gán cho ai trước đó. Nên `mask` đủ làm trạng thái.

## 4. Ý tưởng nền tảng / trực giác

Hãy hình dung công nhân đến bãi xe lần lượt. Công nhân `k` chọn một xe còn trống `j`: trả `cost[k][j]`, rồi đệ quy cho công nhân `k+1` với tập xe trống nhỏ hơn. Nhiều thứ tự chọn của các công nhân trước dẫn tới **cùng một tập xe đã dùng**, nên ta chỉ cần nhớ **chi phí nhỏ nhất** để đạt mỗi tập đó.

## 5. Brute force — thử mọi chỉnh hợp

**Ý tưởng:** `itertools.permutations(range(m), n)` sinh mọi cách chọn `n` xe có thứ tự (xe thứ `i` cho công nhân `i`); tính tổng chi phí và lấy nhỏ nhất. Đúng theo định nghĩa, dùng làm **oracle** (`m ≤ 7`).

In [3]:
def assign_brute(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    return min(sum(cost[i][sigma[i]] for i in range(n)) for sigma in itertools.permutations(range(m), n))


for w, b, want in examples:
    assert assign_brute(w, b) == want
print([assign_brute(w, b) for w, b, _ in examples])

[6, 4, 4995]


In [4]:
# Code nộp LeetCode cho Brute force (đúng nhưng chậm khi m = 10, n = 10: 10! = 3.6 triệu hoán vị)
class SolutionBrute:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)
        best = float("inf")
        for sigma in itertools.permutations(range(m), n):
            total = 0
            for i in range(n):
                total += abs(workers[i][0] - bikes[sigma[i]][0]) + abs(workers[i][1] - bikes[sigma[i]][1])
            best = min(best, total)
        return best

**Độ phức tạp:** `O(m!/(m-n)! · n)` thời gian, `O(n)` bộ nhớ.

**Nút thắt:** nhiều hoán vị chia sẻ cùng **tập xe đã dùng** ở tiền tố nhưng ta tính lại.

## 6. Bước đột phá

Gán công nhân theo thứ tự và nhớ `dp[mask]` = chi phí nhỏ nhất để gán `popcount(mask)` công nhân đầu vào đúng tập xe `mask`. Chuyển: `dp[mask | 1<<j] = min(dp[mask] + cost[popcount(mask)][j])` với `j ∉ mask`. Đáp án là `min dp[mask]` trên mọi `mask` có `popcount = n`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** quay lui cắt nhánh, **Cách 3** DP bitmask (đáp án chính), **Cách 4** đệ quy có nhớ, **Cách 5** thuật toán Hungarian.

### Cách 2 — Quay lui + cắt nhánh (branch and bound)

**Giải thích:** đệ quy gán công nhân `i`; nếu tổng hiện tại `≥ best` thì **cắt**. Không đảm bảo đa thức nhưng nhanh với dữ liệu thường.

In [5]:
def assign_backtrack(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    best = [float("inf")]
    used = [False] * m

    def dfs(i, cur):
        if cur >= best[0]:
            return                                          # cắt nhánh: đã không tốt hơn
        if i == n:
            best[0] = cur
            return
        for j in range(m):
            if not used[j]:
                used[j] = True
                dfs(i + 1, cur + cost[i][j])
                used[j] = False

    dfs(0, 0)
    return best[0]


for w, b, want in examples:
    assert assign_backtrack(w, b) == want
print([assign_backtrack(w, b) for w, b, _ in examples])

[6, 4, 4995]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionBacktrack:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)
        best = [float("inf")]
        used = [False] * m

        def dfs(i, cur):
            if cur >= best[0]:
                return
            if i == n:
                best[0] = cur
                return
            for j in range(m):
                if not used[j]:
                    used[j] = True
                    d = abs(workers[i][0] - bikes[j][0]) + abs(workers[i][1] - bikes[j][1])
                    dfs(i + 1, cur + d)
                    used[j] = False

        dfs(0, 0)
        return best[0]

**Độ phức tạp:** trường hợp xấu `O(m!/(m-n)!)`, thường nhanh hơn nhiều nhờ cắt nhánh; `O(n)` bộ nhớ.

### Cách 3 — DP bitmask (đáp án chính)

**Ý tưởng:** `dp[mask]` như mục 6. Duyệt `mask` tăng dần (mọi chuyển đi tới mask lớn hơn). Khởi tạo `dp[0] = 0`, còn lại `INF`.

In [7]:
def assign_dp(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    dp[0] = 0
    best = INF
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")                            # công nhân kế tiếp
        if k == n:
            best = min(best, dp[mask])
            continue
        for j in range(m):
            if not mask >> j & 1:
                nm = mask | 1 << j
                v = dp[mask] + cost[k][j]
                if v < dp[nm]:
                    dp[nm] = v
    return best


for w, b, want in examples:
    assert assign_dp(w, b) == want
print([assign_dp(w, b) for w, b, _ in examples])

[6, 4, 4995]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionDP:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)
        INF = float("inf")
        dp = [INF] * (1 << m)
        dp[0] = 0
        best = INF
        for mask in range(1 << m):
            if dp[mask] == INF:
                continue
            k = bin(mask).count("1")
            if k == n:
                best = min(best, dp[mask])
                continue
            for j in range(m):
                if not mask >> j & 1:
                    d = abs(workers[k][0] - bikes[j][0]) + abs(workers[k][1] - bikes[j][1])
                    if dp[mask] + d < dp[mask | 1 << j]:
                        dp[mask | 1 << j] = dp[mask] + d
        return best

**Độ phức tạp:** `O(2^m · m)` thời gian (`≈ 10^4` bước với `m = 10`), `O(2^m)` bộ nhớ.

### Cách 4 — Đệ quy có nhớ

**Ý tưởng:** `go(k, mask)` = chi phí nhỏ nhất để gán công nhân `k..n-1` khi tập xe đã dùng là `mask`; `go(n, ·) = 0`. Có nhớ bằng `lru_cache`. (Ở đây `k = popcount(mask)` nên chỉ cần `mask`, ta giữ `k` cho dễ đọc.)

In [9]:
def assign_topdown(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)

    @lru_cache(maxsize=None)
    def go(k, mask):
        if k == n:
            return 0
        return min(cost[k][j] + go(k + 1, mask | 1 << j) for j in range(m) if not mask >> j & 1)

    return go(0, 0)


for w, b, want in examples:
    assert assign_topdown(w, b) == want
print([assign_topdown(w, b) for w, b, _ in examples])

[6, 4, 4995]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionTopDown:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)

        @lru_cache(maxsize=None)
        def go(k, mask):
            if k == n:
                return 0
            res = float("inf")
            for j in range(m):
                if not mask >> j & 1:
                    d = abs(workers[k][0] - bikes[j][0]) + abs(workers[k][1] - bikes[j][1])
                    res = min(res, d + go(k + 1, mask | 1 << j))
            return res

        return go(0, 0)

**Độ phức tạp:** `O(2^m · m)` thời gian, `O(2^m)` bộ nhớ (chỉ các trạng thái đạt tới).

### Cách 5 — Thuật toán Hungarian (gán tối ưu đa thức)

**Ý tưởng:** bài toán gán tối thiểu trên đồ thị hai phía có trọng số giải được bằng thuật toán Hungarian (Kuhn–Munkres) với **thế năng** `u` (công nhân), `v` (xe): duy trì `u[i] + v[j] ≤ cost[i][j]` và tìm đường tăng tối thiểu. Phiên bản chữ nhật (`n ≤ m`) thêm từng công nhân một. Độ phức tạp `O(n² · m)`, không phụ thuộc `2^m`.

In [11]:
def hungarian(cost):
    """Gán n hàng cho n trong m cột (n <= m) với tổng chi phí nhỏ nhất. Trả về (tổng, danh sách cột của từng hàng)."""
    n, m = len(cost), len(cost[0])
    INF = float("inf")
    u = [0] * (n + 1)
    v = [0] * (m + 1)
    p = [0] * (m + 1)                                        # p[j] = hàng (1-based) đang gán cho cột j
    way = [0] * (m + 1)
    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = [INF] * (m + 1)
        used = [False] * (m + 1)
        while True:
            used[j0] = True
            i0, delta, j1 = p[j0], INF, 0
            for j in range(1, m + 1):
                if not used[j]:
                    cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j], way[j] = cur, j0
                    if minv[j] < delta:
                        delta, j1 = minv[j], j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:                                          # lần ngược đường tăng
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break
    cols = [0] * n
    for j in range(1, m + 1):
        if p[j]:
            cols[p[j] - 1] = j - 1
    return sum(cost[i][cols[i]] for i in range(n)), cols


def assign_hungarian(workers, bikes):
    return hungarian(cost_matrix(workers, bikes))[0]


for w, b, want in examples:
    assert assign_hungarian(w, b) == want
print([assign_hungarian(w, b) for w, b, _ in examples])

[6, 4, 4995]


In [12]:
# Code nộp LeetCode cho Cách 5 (Hungarian)
class SolutionHungarian:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)
        cost = [[abs(w[0] - b[0]) + abs(w[1] - b[1]) for b in bikes] for w in workers]
        INF = float("inf")
        u, v, p, way = [0] * (n + 1), [0] * (m + 1), [0] * (m + 1), [0] * (m + 1)
        for i in range(1, n + 1):
            p[0] = i
            j0 = 0
            minv = [INF] * (m + 1)
            used = [False] * (m + 1)
            while True:
                used[j0] = True
                i0, delta, j1 = p[j0], INF, 0
                for j in range(1, m + 1):
                    if not used[j]:
                        cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
                        if cur < minv[j]:
                            minv[j], way[j] = cur, j0
                        if minv[j] < delta:
                            delta, j1 = minv[j], j
                for j in range(m + 1):
                    if used[j]:
                        u[p[j]] += delta
                        v[j] -= delta
                    else:
                        minv[j] -= delta
                j0 = j1
                if p[j0] == 0:
                    break
            while True:
                j1 = way[j0]
                p[j0] = p[j1]
                j0 = j1
                if j0 == 0:
                    break
        return sum(cost[p[j] - 1][j - 1] for j in range(1, m + 1) if p[j])

**Độ phức tạp:** `O(n² · m)` thời gian, `O(n + m)` bộ nhớ phụ (ngoài ma trận chi phí). Với `n, m ≤ 10` nhanh hơn DP bitmask về số phép tính; DP bitmask đơn giản và đủ cho đề.

## 8. TRACE từng bước

Trace **Cách 3** với ví dụ 2: `workers = [[0,0],[1,1],[2,0]]`, `bikes = [[1,0],[2,2],[2,1]]`. Ma trận chi phí:

| | xe 0 | xe 1 | xe 2 |
|---|---|---|---|
| CN 0 | 1 | 4 | 3 |
| CN 1 | 1 | 2 | 1 |
| CN 2 | 1 | 2 | 1 |

`dp[mask]` (bit `j` = xe `j` đã dùng):

| `mask` (xe đã dùng) | `popcount` | `dp` |
|---|---|---|
| `{}` | 0 | 0 |
| `{0}` | 1 | 1 |
| `{1}` | 1 | 4 |
| `{2}` | 1 | 3 |
| `{0,1}` | 2 | `min(1 + 2, 4 + 1) = 3` |
| `{0,2}` | 2 | `min(1 + 1, 3 + 1) = 2` |
| `{1,2}` | 2 | `min(4 + 1, 3 + 2) = 5` |
| `{0,1,2}` | 3 | `min(dp{0,1} + 1, dp{0,2} + 2, dp{1,2} + 1) = min(4, 4, 6) = 4` |

Đáp án `4`. Hàm trace in từng trạng thái:

In [13]:
def trace_dp(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    dp[0] = 0
    print("cost =", cost)
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        used = [j for j in range(m) if mask >> j & 1]
        print(f"  mask = {mask:0{m}b}, xe đã dùng {used}, đã gán {k} công nhân, dp = {dp[mask]}")
        if k == n:
            continue
        for j in range(m):
            if not mask >> j & 1:
                dp[mask | 1 << j] = min(dp[mask | 1 << j], dp[mask] + cost[k][j])
    ans = min(dp[mask] for mask in range(1 << m) if bin(mask).count("1") == n)
    print("  đáp án =", ans)
    return ans


assert trace_dp(*examples[1][:2]) == 4

cost = [[1, 4, 3], [1, 2, 1], [1, 2, 1]]
  mask = 000, xe đã dùng [], đã gán 0 công nhân, dp = 0
  mask = 001, xe đã dùng [0], đã gán 1 công nhân, dp = 1
  mask = 010, xe đã dùng [1], đã gán 1 công nhân, dp = 4
  mask = 011, xe đã dùng [0, 1], đã gán 2 công nhân, dp = 3
  mask = 100, xe đã dùng [2], đã gán 1 công nhân, dp = 3
  mask = 101, xe đã dùng [0, 2], đã gán 2 công nhân, dp = 2
  mask = 110, xe đã dùng [1, 2], đã gán 2 công nhân, dp = 5
  mask = 111, xe đã dùng [0, 1, 2], đã gán 3 công nhân, dp = 4
  đáp án = 4


**Khôi phục cách gán:** lưu `parent[mask] = (mask_trước, xe)` rồi lần ngược từ `mask` tối ưu. Với ví dụ 2: công nhân 0 – xe 0, công nhân 1 – xe 1 (hoặc xe 2), công nhân 2 – xe 2 (hoặc xe 1), tổng `4`.

In [14]:
def best_assignment(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    par = [None] * (1 << m)
    dp[0] = 0
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            continue
        for j in range(m):
            if not mask >> j & 1 and dp[mask] + cost[k][j] < dp[mask | 1 << j]:
                dp[mask | 1 << j] = dp[mask] + cost[k][j]
                par[mask | 1 << j] = (mask, j)
    end = min((mk for mk in range(1 << m) if bin(mk).count("1") == n), key=lambda mk: dp[mk])
    sigma, mk = [], end
    while mk:
        pm, j = par[mk]
        sigma.append(j)
        mk = pm
    sigma.reverse()
    return dp[end], sigma


val, sigma = best_assignment(*examples[1][:2])
print(val, sigma)
assert val == 4 and len(set(sigma)) == 3 and sum(cost_matrix(*examples[1][:2])[i][sigma[i]] for i in range(3)) == 4

4 [0, 1, 2]


## 9. Code nộp LeetCode

Chọn **Cách 3** (DP bitmask). Giữ nguyên `class Solution`, `assignBikes(self, workers, bikes)`:

In [15]:
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)
        INF = float("inf")
        dp = [INF] * (1 << m)                       # dp[mask] = chi phí nhỏ nhất khi các xe trong mask đã được gán
        dp[0] = 0                                   # cho popcount(mask) công nhân đầu tiên
        best = INF
        for mask in range(1 << m):
            if dp[mask] == INF:
                continue
            k = bin(mask).count("1")                # công nhân kế tiếp cần xe
            if k == n:
                best = min(best, dp[mask])          # đã gán đủ n công nhân
                continue
            for j in range(m):
                if not mask >> j & 1:               # xe j còn trống
                    d = abs(workers[k][0] - bikes[j][0]) + abs(workers[k][1] - bikes[j][1])
                    nm = mask | 1 << j
                    if dp[mask] + d < dp[nm]:
                        dp[nm] = dp[mask] + d
        return best


print([Solution().assignBikes(w, b) for w, b, _ in examples])
assert all(Solution().assignBikes(w, b) == v for w, b, v in examples)

[6, 4, 4995]


## 10. Vì sao đúng

**(a) Mô hình.** Một cách gán hợp lệ là đơn ánh `σ`. Với thứ tự công nhân cố định `0, 1, …, n-1`, `σ` tương ứng một **dãy** các xe `σ(0), σ(1), …` đôi một khác nhau.

**(b) Trạng thái.** Sau khi gán `k` công nhân đầu bằng tập xe `S = {σ(0), …, σ(k-1)}` (`|S| = k`), chi phí **còn lại** chỉ phụ thuộc `S` (xe trống = phần bù của `S`) và các công nhân `k..n-1`; không phụ thuộc xe nào của `S` gán cho ai. Vậy cách gán tối ưu của phần còn lại chỉ phụ thuộc `S` (**cấu trúc con tối ưu**).

**(c) Phương trình.** Đặt `dp[S]` = chi phí nhỏ nhất của việc gán công nhân `0..|S|-1` vào đúng tập xe `S`. `dp[∅] = 0`. Với `S ≠ ∅`, công nhân cuối trong `S` (công nhân `|S|-1`) dùng xe `j ∈ S` và các công nhân trước dùng `S ∖ {j}`: `dp[S] = min_{j ∈ S} dp[S ∖ {j}] + cost[|S|-1][j]`. Mã dùng dạng "đẩy" tương đương: từ `mask` thêm xe `j` cho công nhân `popcount(mask)`.

**(d) Đáp án.** Chọn nhỏ nhất `dp[S]` trên mọi `S` có `|S| = n` (vì `n ≤ m`, không bắt buộc dùng hết xe). Mỗi `mask` đạt được chỉ khi mọi bit trước đó hợp lệ; `INF` bỏ qua trạng thái chưa đạt.

**(e) Thứ tự duyệt.** Chuyển luôn đi tới `mask` **lớn hơn** (thêm một bit) nên duyệt `mask` tăng dần đảm bảo `dp[mask]` đã chốt trước khi dùng.

**(f) Các cách còn lại.** Cách 1 là định nghĩa; Cách 2 cùng cây tìm kiếm với cắt nhánh (cắt `cur ≥ best` không loại nghiệm tối ưu vì chi phí không âm); Cách 4 là bản đệ quy của (c); Cách 5 giải trực tiếp bài toán gán bằng đối ngẫu quy hoạch tuyến tính (Hungarian) nên đúng theo định lý của Kuhn–Munkres.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp vét cạn chỉnh hợp trên mọi cấu hình nhỏ của lưới `3×3` và hàng nghìn cấu hình ngẫu nhiên, và khớp nhau trên `n = m = 10` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(m!/(m-n)! · n)` | `O(n)` | oracle `m ≤ 7` |
| 2. Quay lui cắt nhánh | xấu `O(m!)`, thường nhanh | `O(n)` | dễ viết |
| 3. DP bitmask | `O(2^m · m)` | `O(2^m)` | **đáp án chính** |
| 4. Top-down | `O(2^m · m)` | `O(2^m)` | đệ quy dễ đọc |
| 5. Hungarian | `O(n² m)` | `O(n + m)` | mở rộng tới `n, m` lớn |

Đo thực tế (`n = m = 10`, toạ độ ngẫu nhiên):

In [16]:
def gen(rng, n, m, span):
    cells = rng.sample([(x, y) for x in range(span) for y in range(span)], n + m)
    return [list(c) for c in cells[:n]], [list(c) for c in cells[n:]]


def bench(fn, *args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1066)
w = [[rng.randrange(1000), rng.randrange(1000)] for _ in range(10)]
b = [[rng.randrange(1000), rng.randrange(1000)] for _ in range(10)]
for name, fn in [("backtrack", assign_backtrack), ("dp", assign_dp), ("topdown", assign_topdown), ("hungarian", assign_hungarian)]:
    print(f"{name:10s}: {bench(fn, w, b) * 1000:8.1f} ms -> {fn(w, b)}")

backtrack :    289.1 ms -> 3203
dp        :      1.1 ms -> 3203
topdown   :      1.8 ms -> 3203
hungarian :      0.1 ms -> 3203


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi cấu hình nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "backtrack": assign_backtrack,
    "dp": assign_dp,
    "topdown": assign_topdown,
    "hungarian": assign_hungarian,
    "Solution": lambda w, b: Solution().assignBikes(w, b),
    "SolutionBacktrack": lambda w, b: SolutionBacktrack().assignBikes(w, b),
    "SolutionDP": lambda w, b: SolutionDP().assignBikes(w, b),
    "SolutionTopDown": lambda w, b: SolutionTopDown().assignBikes(w, b),
    "SolutionHungarian": lambda w, b: SolutionHungarian().assignBikes(w, b),
}
BRUTE = {"brute": assign_brute, "SolutionBrute": lambda w, b: SolutionBrute().assignBikes(w, b)}


def check(w, b, want=None, brute=False):
    if want is None:
        want = assign_brute(w, b) if brute else Solution().assignBikes(w, b)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        if len(b) > 8 and "acktrack" in name:
            continue                                        # quay lui cắt nhánh có thể rất chậm khi m = 9..10
        got = fn([p[:] for p in w], [p[:] for p in b])
        assert got == want, (name, w, b, got, want)
    return want


for w, b, want in examples:
    check(w, b, want, brute=len(b) <= 7)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn (ví dụ 3 có m = 5)")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn (ví dụ 3 có m = 5)


### B. Edge case

- `n = m = 1`: đáp án là khoảng cách giữa hai điểm.
- `n = 1`, `m = 10`: chọn xe gần nhất.
- `n = m` và `n < m` (không bắt buộc dùng hết xe).
- Công nhân và xe thẳng hàng (ví dụ 3): gán "song song" tối ưu.
- Gán tham lam gần nhất **sai** (mục 13): bố cục "xe gần nhất cho cả hai công nhân".
- Toạ độ lớn `999` (tổng tới ~ `20 000`).
- Công nhân đứng sát nhau, xe đứng sát nhau.

In [18]:
edge = [
    ([[0, 0]], [[3, 4]], 7),
    ([[0, 0]], [[9, 9], [1, 0], [5, 5], [0, 2]], 1),
    ([[0, 0], [1, 0]], [[2, 0], [3, 0]], 4),
    ([[0, 0], [10, 0]], [[1, 0], [9, 0]], 2),
    ([[0, 0], [2, 0]], [[1, 0], [5, 0]], 4),
    ([[999, 999], [0, 0]], [[0, 1], [998, 999]], 2),
    ([[0, 0], [1, 1]], [[500, 500], [501, 501], [0, 1]], None),
    ([[x, 0] for x in range(7)], [[x, 3] for x in range(7)], 21),
    ([[0, 0], [0, 1], [0, 2]], [[5, 0], [5, 1], [5, 2], [1, 0]], None),
]
for w, b, want in edge:
    got = check(w, b, want, brute=len(b) <= 7)
    print(f"  n = {len(w)}, m = {len(b)}: {got}")
print(len(edge), "edge case: mọi cài đặt khớp")

  n = 1, m = 1: 7
  n = 1, m = 4: 1
  n = 2, m = 2: 4
  n = 2, m = 2: 2
  n = 2, m = 2: 4
  n = 2, m = 2: 2
  n = 2, m = 3: 999
  n = 7, m = 7: 21
  n = 3, m = 4: 11
9 edge case: mọi cài đặt khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_greedy_worker(workers, bikes):                  # mỗi công nhân (theo thứ tự) lấy xe trống gần nhất
    used, total = set(), 0
    for w in workers:
        j = min((j for j in range(len(bikes)) if j not in used), key=lambda j: manhattan(w, bikes[j]))
        used.add(j)
        total += manhattan(w, bikes[j])
    return total


def bug_greedy_global(workers, bikes):                  # kiểu Campus Bikes I: cặp (công nhân, xe) gần nhất toàn cục trước
    pairs = sorted((manhattan(w, b), i, j) for i, w in enumerate(workers) for j, b in enumerate(bikes))
    uw, ub, total = set(), set(), 0
    for d, i, j in pairs:
        if i not in uw and j not in ub:
            uw.add(i)
            ub.add(j)
            total += d
    return total


def bug_reuse_bike(workers, bikes):                     # cho phép nhiều công nhân dùng cùng một xe
    return sum(min(manhattan(w, b) for b in bikes) for w in workers)


def bug_need_all_bikes(workers, bikes):                 # đòi dùng hết mọi xe (chỉ đúng khi n == m)
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    dp[0] = 0
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k >= n:
            continue
        for j in range(m):
            if not mask >> j & 1:
                dp[mask | 1 << j] = min(dp[mask | 1 << j], dp[mask] + cost[k][j])
    return dp[(1 << m) - 1]


def bug_chebyshev(workers, bikes):                      # nhầm khoảng cách: max(|dx|, |dy|)
    cost = [[max(abs(w[0] - b[0]), abs(w[1] - b[1])) for b in bikes] for w in workers]
    return min(sum(cost[i][s[i]] for i in range(len(workers))) for s in itertools.permutations(range(len(bikes)), len(workers)))


def bug_small_inf(workers, bikes):                      # INF = 1000 nhỏ hơn đáp án thực
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = 1000
    dp = [INF] * (1 << m)
    dp[0] = 0
    best = INF
    for mask in range(1 << m):
        if dp[mask] >= INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            best = min(best, dp[mask])
            continue
        for j in range(m):
            if not mask >> j & 1:
                dp[mask | 1 << j] = min(dp[mask | 1 << j], dp[mask] + cost[k][j])
    return best


BUGS = [bug_greedy_worker, bug_greedy_global, bug_reuse_bike, bug_need_all_bikes, bug_chebyshev]
rng = random.Random(10660)
pool = []
for _ in range(1500):
    n = rng.randint(1, 4)
    m = rng.randint(n, 5)
    pool.append(gen(rng, n, m, 5))
for bug in BUGS:
    found = next(((w, b) for w, b in pool if bug(w, b) != assign_brute(w, b)), None)
    assert found, bug.__name__
    w, b = found
    print(f"{bug.__name__:20s} sai trên workers = {w}, bikes = {b}: ra {bug(w, b)}, đúng {assign_brute(w, b)}")
w, b = [[0, 0]], [[999, 999]]
print("bug_small_inf: ra", bug_small_inf(w, b), "| đúng", Solution().assignBikes(w, b))
assert bug_small_inf(w, b) == 1000 and Solution().assignBikes(w, b) == 1998

bug_greedy_worker    sai trên workers = [[2, 2], [0, 3], [0, 1]], bikes = [[1, 0], [3, 4], [4, 1]]: ra 11, đúng 9
bug_greedy_global    sai trên workers = [[4, 1], [4, 0], [1, 2], [4, 3]], bikes = [[0, 1], [2, 3], [0, 4], [2, 4]]: ra 17, đúng 15
bug_reuse_bike       sai trên workers = [[0, 0], [1, 3], [1, 2], [2, 4]], bikes = [[0, 1], [4, 2], [0, 3], [4, 1], [3, 4]]: ra 5, đúng 6
bug_need_all_bikes   sai trên workers = [[3, 2]], bikes = [[1, 4], [1, 3], [2, 0], [0, 3]]: ra inf, đúng 3
bug_chebyshev        sai trên workers = [[3, 2]], bikes = [[1, 4], [1, 3], [2, 0], [0, 3]]: ra 2, đúng 3
bug_small_inf: ra 1000 | đúng 1998


### D. Vét cạn mọi cấu hình nhỏ

Trên lưới `3×3` (`9` ô), chọn **tập** `n` ô cho công nhân và **tập** `m` ô còn lại làm xe, rồi thử mọi **thứ tự** cố định (thứ tự công nhân theo chỉ số): `(n, m) ∈ {(1,1), (1,2), (2,2), (2,3), (3,3)}`. Với mỗi cấu hình so vét cạn với mọi cài đặt.

In [20]:
t0 = time.perf_counter()
cells = [[x, y] for x in range(3) for y in range(3)]
cnt = 0
for n, m in [(1, 1), (1, 2), (2, 2), (2, 3), (3, 3)]:
    for wi in itertools.combinations(range(9), n):
        rest = [c for c in range(9) if c not in wi]
        for bi in itertools.combinations(rest, m):
            w = [cells[i] for i in wi]
            b = [cells[i] for i in bi]
            want = assign_brute(w, b)
            for name, fn in IMPLS.items():
                assert fn(w, b) == want, (name, w, b)
            cnt += 1
print(cnt, "cấu hình:", len(IMPLS), "cài đặt khớp vét cạn;", round(time.perf_counter() - t0, 1), "s")

4020 cấu hình: 9 cài đặt khớp vét cạn; 0.3 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) `m ≤ 7`: mọi cách khớp vét cạn; (2) `m ≤ 10`: các cách khớp nhau (DP, top-down, Hungarian; quay lui chỉ khi `m ≤ 8`); (3) **tịnh tiến** mọi toạ độ không đổi đáp án; (4) **hoán đổi `x ↔ y`** không đổi; (5) **hoán vị** thứ tự công nhân hoặc xe không đổi; (6) **thêm một xe** không làm đáp án tăng; (7) cận: `Σ_i min_j cost[i][j] ≤ ans ≤ Σ_i cost[i][i]` (gán đồng nhất); (8) khôi phục cách gán hợp lệ và có chi phí bằng đáp án; (9) Hungarian trả về một đơn ánh có chi phí bằng đáp án.

In [21]:
rng = random.Random(20240166)
t0 = time.perf_counter()
for it in range(300):
    n = rng.randint(1, 6)
    m = rng.randint(n, 7)
    w, b = gen(rng, n, m, rng.choice([4, 10, 1000]))
    check(w, b, brute=True)                                                           # (1)
print("(1) 300 cấu hình m <= 7 khớp vét cạn:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(120):
    n = rng.randint(1, 10)
    m = rng.randint(n, 10)
    w, b = gen(rng, n, m, rng.choice([5, 12, 1000]))
    ans = check(w, b)                                                                 # (2)
    dx, dy = rng.randint(0, 5), rng.randint(0, 5)
    sh = lambda ps: [[x + dx, y + dy] for x, y in ps]
    assert Solution().assignBikes(sh(w), sh(b)) == ans                                # (3)
    sw = lambda ps: [[y, x] for x, y in ps]
    assert Solution().assignBikes(sw(w), sw(b)) == ans                                # (4)
    w2, b2 = w[:], b[:]
    rng.shuffle(w2)
    rng.shuffle(b2)
    assert Solution().assignBikes(w2, b2) == ans                                      # (5)
    extra = None
    for _ in range(50):
        c = [rng.randrange(0, 1000), rng.randrange(0, 1000)]
        if c not in w and c not in b:
            extra = c
            break
    if extra is not None and m < 10:
        assert Solution().assignBikes(w, b + [extra]) <= ans                          # (6)
    cost = cost_matrix(w, b)
    assert sum(min(r) for r in cost) <= ans <= sum(cost[i][i] for i in range(n))      # (7)
    val, sigma = best_assignment(w, b)
    assert val == ans and len(set(sigma)) == n and sum(cost[i][sigma[i]] for i in range(n)) == ans   # (8)
    hv, cols = hungarian(cost)
    assert hv == ans and len(set(cols)) == n and sum(cost[i][cols[i]] for i in range(n)) == ans      # (9)
print("(2)-(9) 120 cấu hình m <= 10:", round(time.perf_counter() - t0, 1), "s")

(1) 300 cấu hình m <= 7 khớp vét cạn: 10.7 s


(2)-(9) 120 cấu hình m <= 10: 4.5 s


### F. Giới hạn

`n = m = 10`: toạ độ ngẫu nhiên tới `999`; thẳng hàng (như ví dụ 3); cụm điểm; xe ở rất xa; `n = 1, m = 10`; `n = 5, m = 10`. Các cách khớp nhau và chạy nhanh.

In [22]:
t0 = time.perf_counter()
rg = random.Random(31)
big = []
for _ in range(15):
    big.append(([[rg.randrange(1000), rg.randrange(1000)] for _ in range(10)], [[rg.randrange(1000), rg.randrange(1000)] for _ in range(10)]))
big.append(([[i, 0] for i in range(10)], [[i, 999] for i in range(10)]))
big.append(([[i, i] for i in range(10)], [[999 - i, 999 - i] for i in range(10)]))
big.append(([[rg.randrange(40), rg.randrange(40)] for _ in range(5)], [[500 + rg.randrange(40), 500 + rg.randrange(40)] for _ in range(10)]))
seen_ok = 0
for w, b in big:
    if len({tuple(p) for p in w + b}) != len(w) + len(b):
        continue                                        # bỏ qua toạ độ trùng (đề đảm bảo phân biệt)
    check(w, b)
    seen_ok += 1
print(seen_ok, "cấu hình lớn (n, m <= 10): mọi cách khớp nhau;", round(time.perf_counter() - t0, 1), "s")
assert Solution().assignBikes([[i, 0] for i in range(10)], [[i, 999] for i in range(10)]) == 9990

18 cấu hình lớn (n, m <= 10): mọi cách khớp nhau; 0.4 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tham lam theo từng công nhân (lấy xe gần nhất còn trống)

In [23]:
w, b = [[0, 0], [1, 0]], [[2, 0], [0, 1]]
print("bug:", bug_greedy_worker(w, b), "| đúng:", Solution().assignBikes(w, b))
w, b = next((w, b) for w, b in pool if bug_greedy_worker(w, b) != assign_brute(w, b))
print("phản ví dụ:", w, b, "| bug:", bug_greedy_worker(w, b), "| đúng:", assign_brute(w, b))
assert bug_greedy_worker(w, b) > assign_brute(w, b)

bug: 2 | đúng: 2
phản ví dụ: [[2, 2], [0, 3], [0, 1]] [[1, 0], [3, 4], [4, 1]] | bug: 11 | đúng: 9


**Vì sao sai:** công nhân đầu "cướp" xe gần mình nhưng đó có thể là xe duy nhất gần công nhân sau; tối ưu **toàn cục** cần cân nhắc cả hai.

### Lỗi 2: tham lam cặp gần nhất toàn cục (Campus Bikes I)

In [24]:
w, b = next((w, b) for w, b in pool if bug_greedy_global(w, b) != assign_brute(w, b))
print("phản ví dụ:", w, b, "| bug:", bug_greedy_global(w, b), "| đúng:", assign_brute(w, b))
assert bug_greedy_global(w, b) > assign_brute(w, b)

phản ví dụ: [[4, 1], [4, 0], [1, 2], [4, 3]] [[0, 1], [2, 3], [0, 4], [2, 4]] | bug: 17 | đúng: 15


**Vì sao sai:** bài I yêu cầu **quy tắc gán tham lam** theo đề; bài II yêu cầu **tối thiểu hoá tổng**, tham lam không đảm bảo tối ưu.

### Lỗi 3: nhiều công nhân dùng chung một xe

In [25]:
w, b = [[0, 0], [0, 1]], [[1, 0], [9, 9]]
print("bug:", bug_reuse_bike(w, b), "| đúng:", Solution().assignBikes(w, b))
assert bug_reuse_bike(w, b) < Solution().assignBikes(w, b)

bug: 3 | đúng: 18


**Vì sao sai:** mỗi xe chỉ giao cho **một** công nhân. Lấy `min` độc lập cho từng người bỏ qua ràng buộc này nên cho đáp án **quá nhỏ**. DP bitmask chính là cách ghi nhớ xe đã dùng.

### Lỗi 4: đòi dùng hết mọi xe khi `n < m`

In [26]:
w, b = [[0, 0]], [[1, 0], [5, 5]]
print("bug:", bug_need_all_bikes(w, b), "| đúng:", Solution().assignBikes(w, b))
assert bug_need_all_bikes(w, b) == float("inf") and Solution().assignBikes(w, b) == 1

bug: inf | đúng: 1


**Vì sao sai:** chỉ có `n` công nhân nên chỉ `n` xe được dùng; đáp án là **min** `dp[mask]` trên mọi `mask` có đúng `n` bit, không phải `dp[full]`.

### Lỗi 5: nhầm khoảng cách

In [27]:
w, b = next((w, b) for w, b in pool if bug_chebyshev(w, b) != assign_brute(w, b))
print("phản ví dụ:", w, b, "| bug:", bug_chebyshev(w, b), "| đúng:", assign_brute(w, b))
assert bug_chebyshev(w, b) < assign_brute(w, b)

phản ví dụ: [[3, 2]] [[1, 4], [1, 3], [2, 0], [0, 3]] | bug: 2 | đúng: 3


**Vì sao sai:** khoảng cách là **Manhattan** `|dx| + |dy|`, không phải `max(|dx|, |dy|)` hay bình phương Euclid.

### Lỗi 6 (bonus): hằng `INF` quá nhỏ

In [28]:
w, b = [[0, 0]], [[999, 999]]
print("bug:", bug_small_inf(w, b), "| đúng:", Solution().assignBikes(w, b))
assert bug_small_inf(w, b) == 1000 and Solution().assignBikes(w, b) == 1998

bug: 1000 | đúng: 1998


**Vì sao sai:** khoảng cách một cặp có thể tới `1998` và tổng tới ~`20 000`; `INF = 1000` làm trạng thái đúng bị coi là không tới được. Dùng `float("inf")` hoặc `10**9`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "gán `n` đối tượng cho `m ≥ n` đối tượng, mỗi cái dùng một lần, tối thiểu chi phí" với `m ≤ ~20` → **DP bitmask**; `m` lớn → **Hungarian / min-cost flow**.

**Mẹo:**

1. Xử lý đối tượng theo thứ tự cố định, trạng thái là **tập đã dùng**; `k = popcount(mask)`.
2. Đáp án là min trên mọi `mask` có đúng `n` bit khi `n < m`.
3. `INF` lớn (`float("inf")`), kiểm tra `dp[mask] == INF` để bỏ qua.
4. Chi phí không âm ⇒ cắt nhánh `cur ≥ best` an toàn.
5. Tham lam thường **sai** trong bài gán tối ưu; dùng brute để chứng minh bằng phản ví dụ.
6. Oracle: `itertools.permutations(range(m), n)` cho `m ≤ 7`.

**Họ bài cùng pattern:** Campus Bikes (1057), Minimum Cost to Connect Two Groups of Points (1595), Fair Distribution of Cookies (2305), Partition to K Equal Sum Subsets (698), Maximum Compatibility Score Sum (1947), Find Minimum Time to Finish All Jobs (1723).

## 15. Biến thể

1. **Khôi phục cách gán** (đã làm ở mục 8): `best_assignment`.
2. **Tối đa hoá** tổng khoảng cách (cùng DP, đổi `min` thành `max`).
3. **Tối thiểu hoá khoảng cách lớn nhất** (bottleneck): `min` của `max` thay vì tổng.
4. **Đếm số cách gán tối ưu** (đếm song song với `dp`).

Cài đặt và đối chiếu vét cạn:

In [29]:
def assign_max(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    NEG = -1
    dp = [NEG] * (1 << m)
    dp[0] = 0
    best = NEG
    for mask in range(1 << m):
        if dp[mask] < 0:
            continue
        k = bin(mask).count("1")
        if k == n:
            best = max(best, dp[mask])
            continue
        for j in range(m):
            if not mask >> j & 1:
                dp[mask | 1 << j] = max(dp[mask | 1 << j], dp[mask] + cost[k][j])
    return best


def assign_bottleneck(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    dp[0] = 0
    best = INF
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            best = min(best, dp[mask])
            continue
        for j in range(m):
            if not mask >> j & 1:
                dp[mask | 1 << j] = min(dp[mask | 1 << j], max(dp[mask], cost[k][j]))
    return best


def assign_count(workers, bikes):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    cnt = [0] * (1 << m)
    dp[0], cnt[0] = 0, 1
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            continue
        for j in range(m):
            if not mask >> j & 1:
                nm, v = mask | 1 << j, dp[mask] + cost[k][j]
                if v < dp[nm]:
                    dp[nm], cnt[nm] = v, cnt[mask]
                elif v == dp[nm]:
                    cnt[nm] += cnt[mask]
    full = [mk for mk in range(1 << m) if bin(mk).count("1") == n]
    top = min(dp[mk] for mk in full)
    return top, sum(cnt[mk] for mk in full if dp[mk] == top)


rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 5)
    m = rng.randint(n, 6)
    w, b = gen(rng, n, m, rng.choice([4, 5, 6]))
    cost = cost_matrix(w, b)
    perms = list(itertools.permutations(range(m), n))
    sums = [sum(cost[i][s[i]] for i in range(n)) for s in perms]
    assert assign_max(w, b) == max(sums)
    assert assign_bottleneck(w, b) == min(max(cost[i][s[i]] for i in range(n)) for s in perms)
    assert assign_count(w, b) == (min(sums), sums.count(min(sums)))
print("tối đa hoá / bottleneck / đếm số cách gán tối ưu: khớp vét cạn")

tối đa hoá / bottleneck / đếm số cách gán tối ưu: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi chỉnh hợp | `O(m!/(m-n)!)` | `O(n)` | oracle |
| 2. Quay lui cắt nhánh | `cur ≥ best` thì dừng | xấu `O(m!)` | `O(n)` | dữ liệu thường |
| 3. DP bitmask | `dp[mask]` | `O(2^m · m)` | `O(2^m)` | **đáp án chính** |
| 4. Top-down | `go(k, mask)` | `O(2^m · m)` | `O(2^m)` | dễ đọc |
| 5. Hungarian | thế năng + đường tăng | `O(n² m)` | `O(n + m)` | `n, m` lớn |

Mọi cách khớp vét cạn trên mọi cấu hình nhỏ của lưới `3×3` và `300` cấu hình ngẫu nhiên `m ≤ 7`; khớp nhau trên `m ≤ 10`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().assignBikes([[0, 0]], [[1, 1]]), Solution().assignBikes([[0, 0], [1, 0]], [[0, 1], [1, 1]]), Solution().assignBikes([[0, 0], [10, 0]], [[1, 0], [9, 0]]), Solution().assignBikes([[0, 0]], [[5, 0], [0, 2]]))

2 2 2 2


### Bài 2 (tính tay)

`workers = [[0,0],[2,0]]`, `bikes = [[1,0],[3,0],[0,5]]`. Lập ma trận chi phí `2×3` và bảng `dp[mask]` (với `mask` có `0`, `1`, `2` bit). Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP bitmask:

In [31]:
def assign_todo(workers, bikes):
    n, m = len(workers), len(bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    dp[0] = 0
    best = INF
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            pass        # TODO 1: cập nhật best và tiếp tục
        for j in range(m):
            pass        # TODO 2: nếu xe j trống, dp[mask | 1 << j] = min(..., dp[mask] + khoảng cách(workers[k], bikes[j]))
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(assign_todo(w, b) == v for w, b, v in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **đệ quy có nhớ** `go(k, mask)`:

In [32]:
class MySolution:
    def assignBikes(self, workers, bikes):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().assignBikes(w, b) == v for w, b, v in examples)

### Bài 5 (mini challenge): gán sao cho tổng lớn nhất không vượt `T`

Viết `can_assign(workers, bikes, T)` trả về `True` nếu tồn tại cách gán mà **khoảng cách lớn nhất** của một cặp `≤ T` (bài toán quyết định cho bottleneck). Gợi ý: chỉ cho phép các cặp `cost ≤ T` rồi tìm khớp đủ `n` công nhân (bitmask hoặc quay lui).

In [33]:
def can_assign(workers, bikes, T):
    pass            # TODO


# Khi làm xong:
# assert can_assign([[0, 0], [10, 0]], [[1, 0], [9, 0]], 1) is True
# assert can_assign([[0, 0], [1, 0]], [[0, 1], [9, 9]], 2) is False

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `cost[i][j]` là Manhattan.
- Mức 2: `[[0,0],[1,0]]` với xe `[[0,1],[1,1]]`: mỗi công nhân kề một xe cách `1`.
- Mức 3: `2 2 2 2`.

**Bài 2**

- Mức 1: `cost = [[1, 3, 5], [1, 1, 7]]`.
- Mức 2: `dp{0} = 1`, `dp{1} = 3`, `dp{2} = 5`; `dp{0,1} = min(1 + 1, 3 + 1) = 2`; `dp{0,2} = min(1 + 7, 5 + 1) = 6`; `dp{1,2} = min(3 + 7, 5 + 1) = 6`.
- Mức 3: đáp án `2` (công nhân 0 – xe 0, công nhân 1 – xe 1).

**Bài 3**

- Mức 1: nếu `k == n`: `best = min(best, dp[mask])`, `continue`.
- Mức 2: `if not mask >> j & 1:` kiểm tra xe trống.
- Mức 3: `d = abs(...) + abs(...)`; `dp[mask | 1 << j] = min(dp[mask | 1 << j], dp[mask] + d)`.

**Bài 4**

- Mức 1: `go(k, mask)` với `k` là công nhân kế tiếp.
- Mức 2: `go(n, mask) = 0`; trường hợp khác lấy `min` trên xe trống `j` của `d(k, j) + go(k + 1, mask | 1 << j)`.
- Mức 3: dùng `lru_cache`; trả `go(0, 0)`.

**Bài 5**

- Mức 1: lọc các cặp `cost[i][j] ≤ T`.
- Mức 2: DP bitmask theo thứ tự công nhân: `reach[mask]` đúng/sai.
- Mức 3: `reach[mask | 1 << j] |= reach[mask] and cost[k][j] <= T`; đáp án là có `mask` đủ `n` bit với `reach`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().assignBikes([[0, 0]], [[1, 1]]), Solution().assignBikes([[0, 0], [1, 0]], [[0, 1], [1, 1]]), Solution().assignBikes([[0, 0], [10, 0]], [[1, 0], [9, 0]]), Solution().assignBikes([[0, 0]], [[5, 0], [0, 2]]))
print(vals)
assert vals == (2, 2, 2, 2)
assert vals == (assign_brute([[0, 0]], [[1, 1]]), assign_brute([[0, 0], [1, 0]], [[0, 1], [1, 1]]), assign_brute([[0, 0], [10, 0]], [[1, 0], [9, 0]]), assign_brute([[0, 0]], [[5, 0], [0, 2]]))
print("Bài 1 OK")

(2, 2, 2, 2)
Bài 1 OK


### Bài 2

In [35]:
w2, b2 = [[0, 0], [2, 0]], [[1, 0], [3, 0], [0, 5]]
assert cost_matrix(w2, b2) == [[1, 3, 5], [1, 1, 7]]
assert trace_dp(w2, b2) == 2 == assign_brute(w2, b2)
print("Bài 2 OK")

cost = [[1, 3, 5], [1, 1, 7]]
  mask = 000, xe đã dùng [], đã gán 0 công nhân, dp = 0
  mask = 001, xe đã dùng [0], đã gán 1 công nhân, dp = 1
  mask = 010, xe đã dùng [1], đã gán 1 công nhân, dp = 3
  mask = 011, xe đã dùng [0, 1], đã gán 2 công nhân, dp = 2
  mask = 100, xe đã dùng [2], đã gán 1 công nhân, dp = 5
  mask = 101, xe đã dùng [0, 2], đã gán 2 công nhân, dp = 6
  mask = 110, xe đã dùng [1, 2], đã gán 2 công nhân, dp = 6
  đáp án = 2
Bài 2 OK


### Bài 3

In [36]:
def assign_answer(workers, bikes):
    n, m = len(workers), len(bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)
    dp[0] = 0
    best = INF
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            best = min(best, dp[mask])
            continue
        for j in range(m):
            if not mask >> j & 1:
                d = abs(workers[k][0] - bikes[j][0]) + abs(workers[k][1] - bikes[j][1])
                dp[mask | 1 << j] = min(dp[mask | 1 << j], dp[mask] + d)
    return best


assert all(assign_answer(w, b) == v for w, b, v in examples)
for w, b in pool[:500]:
    assert assign_answer(w, b) == assign_brute(w, b)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def assignBikes(self, workers, bikes):
        n, m = len(workers), len(bikes)

        @lru_cache(maxsize=None)
        def go(k, mask):
            if k == n:
                return 0
            return min(abs(workers[k][0] - bikes[j][0]) + abs(workers[k][1] - bikes[j][1]) + go(k + 1, mask | 1 << j)
                       for j in range(m) if not mask >> j & 1)

        return go(0, 0)


assert all(MySolutionAnswer().assignBikes(w, b) == v for w, b, v in examples)
for w, b in pool[:500]:
    assert MySolutionAnswer().assignBikes(w, b) == assign_brute(w, b)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def can_assign(workers, bikes, T):
    n, m = len(workers), len(bikes)
    cost = cost_matrix(workers, bikes)
    reach = [False] * (1 << m)
    reach[0] = True
    for mask in range(1 << m):
        if not reach[mask]:
            continue
        k = bin(mask).count("1")
        if k == n:
            return True
        for j in range(m):
            if not mask >> j & 1 and cost[k][j] <= T:
                reach[mask | 1 << j] = True
    return False


assert can_assign([[0, 0], [10, 0]], [[1, 0], [9, 0]], 1) is True
assert can_assign([[0, 0], [1, 0]], [[0, 1], [9, 9]], 2) is False
for w, b in pool[:500]:
    bn = assign_bottleneck(w, b)
    assert can_assign(w, b, bn) is True and (bn == 0 or can_assign(w, b, bn - 1) is False)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (gán `n` đối tượng cho `m ≥ n` đối tượng, mỗi cái một lần, tối thiểu tổng chi phí; `m ≤ ~20`):**

```python
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> int:
        n, m = len(workers), len(bikes)
        INF = float("inf")
        dp = [INF] * (1 << m)
        dp[0] = 0
        best = INF
        for mask in range(1 << m):
            if dp[mask] == INF:
                continue
            k = bin(mask).count("1")
            if k == n:
                best = min(best, dp[mask])
                continue
            for j in range(m):
                if not mask >> j & 1:
                    d = abs(workers[k][0] - bikes[j][0]) + abs(workers[k][1] - bikes[j][1])
                    dp[mask | 1 << j] = min(dp[mask | 1 << j], dp[mask] + d)
        return best
```

**Checklist trước khi submit:**

- [ ] `k = popcount(mask)` là công nhân kế tiếp; dừng khi `k == n`?
- [ ] Đáp án là min trên mọi `mask` có đúng `n` bit (không phải `dp[full]`)?
- [ ] `INF` đủ lớn (`float("inf")`), bỏ qua `dp[mask] == INF`?
- [ ] Khoảng cách là Manhattan `|dx| + |dy|`?
- [ ] Mỗi xe một người: dùng bit kiểm tra `not mask >> j & 1`?
- [ ] Đã thử `n = m = 1`, `n = 1, m = 10`, ví dụ `4995`, và tham lam bị phản ví dụ?
- [ ] Đúng tên `class Solution`, `assignBikes(self, workers, bikes)`, không in debug.